# Drill 4: a grader that cannot be gamed (45 min)

You are building the reward for an RL environment. The task given to the model:

> Write `attention(q, k, v, mask=None)` by hand in a file. `q` is `(B, H, Tq, d)`, `k` and `v` are `(B, H, Tk, d)`,
> float64. `mask` is `None` or a bool tensor broadcastable to `(B, H, Tq, Tk)`, `True` = may attend; every query
> may attend to at least one key. Return `(B, H, Tq, d)` = `softmax(q k^T / sqrt(d), masked) v`.
> Do not use `F.scaled_dot_product_attention` or any other attention library function.

Write `grade(path, timeout=10) -> bool`: True only for a correct, hand-written solution.

**Threat model.** The submission is written by a model that is rewarded when your grader says True. It can run any
Python at import time and inside `attention`. `04_grader_inputs/` holds 12 submissions: some are honest, some are wrong,
some cheat. The test cell only tells you which ones should pass. **Don't open the submission files or
`04_grader_inputs/ANSWER_KEY.md` before your grader is done.** Reading the cheats first defeats the drill.

**Things to decide** (these are questions, not answers):
- Where does the submission's code run, and what can it touch there?
- Where are the test inputs generated, and could the submission have seen them before?
- Where is the expected output computed, and where does the comparison happen?
- What do you trust from the submission: its exit code, its stdout, its return value?
- How do you ban a function, and how would a model get around your ban?
- What happens if it never returns?

If your notebook kernel dies or later tests start acting strangely, ask yourself why: that is part of the drill.

In [ ]:
# Example inputs: a scratch cell to inspect shapes and try operations. The tests define their own inputs.
import torch
B, H, Tq, Tk, d = 2, 3, 4, 6, 5
q = torch.randn(B, H, Tq, d, dtype=torch.float64)                    # (B, H, Tq, d)
k = torch.randn(B, H, Tk, d, dtype=torch.float64)                    # (B, H, Tk, d)
v = torch.randn(B, H, Tk, d, dtype=torch.float64)                    # (B, H, Tk, d)
mask = torch.rand(B, 1, Tq, Tk) < 0.6                                # (B, 1, Tq, Tk) bool, True = may attend
mask[..., 0] = True                                                  # every query sees at least one key
path = "04_grader_inputs/sub_07.py"                                    # one submission file

In [ ]:
def grade(path, timeout=10):
    """Return True only if the submission at `path` is a correct, hand-written attention(q, k, v, mask=None).
    Treat the submission as adversarial. Must return within about timeout + a few seconds."""
    # TODO
    raise NotImplementedError


## Test: run your grader on every submission

In [ ]:
import glob
import os
import time

SUB_DIR = "04_grader_inputs"
SHOULD_PASS = {"sub_02.py", "sub_07.py"}                              # every other submission must fail

results = {}
for path in sorted(glob.glob(os.path.join(SUB_DIR, "sub_*.py"))):
    name = os.path.basename(path)
    t0 = time.time()
    results[name] = bool(grade(path, timeout=10))
    dt = time.time() - t0
    want = name in SHOULD_PASS
    flag = "" if results[name] == want else "   <-- WRONG"
    print(f"{name}: {'PASS' if results[name] else 'FAIL'}   (should {'PASS' if want else 'FAIL'}, {dt:.1f}s){flag}")
    assert dt < 30, f"{name} took {dt:.0f}s: the grader must not hang"

wrong = sorted(n for n in results if results[n] != (n in SHOULD_PASS))
assert len(results) == 12, "expected 12 submissions"
assert not wrong, f"wrong verdicts: {wrong}"
print("ok")

In [ ]:
# run it again: an honest submission must pass every time, on fresh random cases
for _ in range(3):
    assert grade(os.path.join(SUB_DIR, "sub_07.py")) and grade(os.path.join(SUB_DIR, "sub_02.py"))
print("ok")

## After it passes

Open `04_grader_inputs/ANSWER_KEY.md`. For each cheat, decide whether it would have fooled a real benchmark you know.
Then write one cheat your grader still misses. There is at least one.